In [31]:
# %pip install imblearn
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report,confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_curve,roc_auc_score
from sklearn.linear_model import ElasticNet,Lasso,Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression, SelectKBest
from scipy import stats
from sklearn.feature_selection import f_regression
from sklearn.datasets import load_breast_cancer
from collections import Counter
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, EditedNearestNeighbours
from sklearn.svm  import SVC
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek
from sklearn.datasets import load_digits
from sklearn.model_selection import KFold, StratifiedGroupKFold, cross_val_score
import kagglehub
from kagglehub import KaggleDatasetAdapter
import graphviz
import math
warnings.filterwarnings('ignore')

HR_comma = pd.read_csv('../../Datasets/fromclass/HR_comma_sep.csv')

df = HR_comma.sample(frac=1, random_state=42).reset_index(drop=True)
df.drop_duplicates(inplace=True)


In [32]:
X = df.drop(['left'], axis=1)
y = df['left']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

bool_cols = X_train.select_dtypes(include='bool').columns
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)


# baseline_model = RandomForestClassifier(
#     random_state=42,
#     n_jobs=-1
# )

# baseline_model.fit(X_train, y_train)
# y_pred = baseline_model.predict(X_test)

# print(classification_report(y_test, y_pred, target_names=['Remained', 'Left']))
# baseline_model.get_params()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

param_dist = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 4],
}

rf = RandomForestClassifier()

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    n_iter=10
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)

Best params: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': None}


In [34]:
best_model = random_search.best_estimator_

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Remained', 'Left']))

              precision    recall  f1-score   support

    Remained       0.98      1.00      0.99      3001
        Left       0.99      0.90      0.95       597

    accuracy                           0.98      3598
   macro avg       0.99      0.95      0.97      3598
weighted avg       0.98      0.98      0.98      3598

